# Parte 2 · Lendo curvas da IRT

**Avaliação com habilidade latente · Instituto Kunumi**

Um parâmetro de cada vez: a dificuldade move a curva, a discriminação muda a inclinação, e as duas juntas decidem quais itens separam modelos fortes de fracos.

**Execute a célula de preparação primeiro.** Depois siga as tarefas em ordem: em cada uma, mude *uma* coisa, observe o resultado e responda *uma* pergunta.

In [ ]:
import subprocess
import sys
from pathlib import Path

# No Colab, baixa o repositório do tutorial; localmente, encontra a raiz a partir desta pasta.
if "google.colab" in sys.modules:
    ROOT = Path("/content/latent-ability-ml")
    if not ROOT.exists():
        subprocess.check_call(["git", "clone", "-q", "https://github.com/manuelfjr/latent-ability-ml.git", str(ROOT)])
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils" / "handson.py").exists())

sys.path.insert(0, str(ROOT))
print("Raiz do projeto:", ROOT)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 110,
    "figure.facecolor": "#F0F0F0",
    "axes.facecolor": "#F0F0F0",
    "savefig.facecolor": "#F0F0F0",
    "axes.edgecolor": "#B4ADA4",
    "axes.labelcolor": "#1C2127",
    "axes.titlecolor": "#1C2127",
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#B4ADA4",
    "grid.alpha": 0.35,
    "xtick.color": "#5E5E5E",
    "ytick.color": "#5E5E5E",
    "text.color": "#1C2127",
    "legend.frameon": False,
    "lines.linewidth": 2.6,
    "axes.prop_cycle": mpl.cycler(color=["#F04E44", "#4392D2", "#2FB7CF", "#62CCA3", "#F0D449"]),
})

## Tarefa 1 · Um item, uma curva (~3 min)

Execute. A curva característica do item (CCI) dá a probabilidade de acerto para cada habilidade θ.

**Pergunta.** Por que a probabilidade é exatamente 0,5 quando a habilidade é igual à dificuldade?

In [ ]:
import numpy as np
from utils.handson import binary_irt_probability

theta = np.linspace(-3, 3, 200)   # habilidade latente
p = binary_irt_probability(theta, difficulty=0.0, discrimination=1.0)

fig, ax = plt.subplots()
ax.plot(theta, p)
ax.set_xlabel("habilidade θ")
ax.set_ylabel("P(acerto)")
ax.set_title("Uma curva característica do item")
plt.show()

print("P(acerto) com θ = 0:", binary_irt_probability(0.0, difficulty=0.0, discrimination=1.0))

*Sua resposta:* 

## Tarefa 2 · A dificuldade move a curva (~3 min)

Execute para ver três itens que só diferem na dificuldade. Depois acrescente `2.0` à lista e execute de novo.

**Pergunta.** Qual item é o mais difícil? Como você percebe isso só olhando?

In [ ]:
dificuldades = [-1.0, 0.0, 1.0]   # <- acrescente 2.0

fig, ax = plt.subplots()
for d in dificuldades:
    ax.plot(theta, binary_irt_probability(theta, difficulty=d, discrimination=1.0), label=f"δ = {d}")
ax.axhline(0.5, color="#B4ADA4", linestyle="--", linewidth=1)
ax.set_xlabel("habilidade θ")
ax.set_ylabel("P(acerto)")
ax.set_title("Mesma forma, posições diferentes")
ax.legend()
plt.show()

*Sua resposta:* 

## Tarefa 3 · A discriminação muda a inclinação (~3 min)

Agora a dificuldade fica fixa em `0` e só a discriminação muda. Execute e depois acrescente `5.0` à lista.

**Pergunta.** Qual curva separa melhor alguém com θ = −0,5 de alguém com θ = 0,5?

In [ ]:
discriminacoes = [0.5, 1.0, 3.0]   # <- acrescente 5.0

fig, ax = plt.subplots()
for a in discriminacoes:
    ax.plot(theta, binary_irt_probability(theta, difficulty=0.0, discrimination=a), label=f"a = {a}")
ax.axvspan(-0.5, 0.5, color="#B4ADA4", alpha=0.25)
ax.set_xlabel("habilidade θ")
ax.set_ylabel("P(acerto)")
ax.set_title("Mesmo ponto central, inclinações diferentes")
ax.legend()
plt.show()

*Sua resposta:* 

## Tarefa 4 · Probabilidades numa tabela (~3 min)

Execute. A tabela mostra a probabilidade de acerto de três respondentes (linhas) em três itens (colunas).

**Pergunta.** Qual item separa melhor o respondente com θ = 0 do respondente com θ = 1?

In [ ]:
import pandas as pd

itens = {                # nome: (dificuldade, discriminação)
    "fácil":   (-1.0, 1.0),
    "médio":   (0.0, 1.0),
    "difícil": (1.0, 2.0),
}
habilidades = np.array([-1.0, 0.0, 1.0])

tabela = pd.DataFrame(
    {nome: binary_irt_probability(habilidades, d, a) for nome, (d, a) in itens.items()},
    index=[f"θ = {t:+.0f}" for t in habilidades],
)
tabela.round(2)

*Sua resposta:* 

## Tarefa 5 · Uma curva ao contrário (~3 min)

Execute. O item tem discriminação negativa.

**Pergunta.** Numa avaliação de modelos, o que significaria um exemplo se comportar assim?

In [ ]:
p = binary_irt_probability(theta, difficulty=0.0, discrimination=-1.5)

fig, ax = plt.subplots()
ax.plot(theta, p)
ax.set_xlabel("habilidade θ")
ax.set_ylabel("P(acerto)")
ax.set_title("Discriminação negativa")
plt.show()

*Sua resposta:* 

## Desafio · Desafio: projete um item (~5 min)

Escolha dificuldade e discriminação para que o item separe um modelo fraco (θ = 0) de um forte (θ = 1): o fraco deve acertar no máximo 30% das vezes e o forte pelo menos 90%.

**Pergunta.** Que combinação funcionou? Por que você precisou mexer nos dois parâmetros?

In [ ]:
dificuldade = 0.0      # <- mude aqui
discriminacao = 1.0    # <- e aqui

fraco, forte = binary_irt_probability(np.array([0.0, 1.0]), dificuldade, discriminacao)
print(f"modelo fraco (θ=0): {fraco:.2f}   modelo forte (θ=1): {forte:.2f}")
print("Conseguiu!" if fraco <= 0.30 and forte >= 0.90 else "Ainda não: busque fraco ≤ 0,30 e forte ≥ 0,90")

*Sua resposta:* 